<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

In [93]:
# %%bash
# !(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
# cd /content && rm -rf /content/rome
# git clone https://github.com/kmeng01/rome rome > install.log 2>&1
# pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&∏
# pip install --upgrade google-cloud-storage >> install.log 2>&1



In [94]:
# %%bash
# pwd
# # Remove any existing 'rome' directory before cloning
# rm -rf /rome
# # Clone the repository into the current directory
# git clone https://github.com/kmeng01/rome rome > install.log 2>&1
# # Install dependencies
# pip install -r ./rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
# # Upgrade Google Cloud Storage package
# pip install --upgrade google-cloud-storage >> install.log 2>&1
# echo "Installation complete. Check install.log for details."

In [1]:
import os
if not(os.path.exists("saved_mlps")):
    os.chdir("rome")
! ls

 baselines	        eval_log-5.log		  notebooks
 CITATION.cff	        experiments		  README.md
 data		        globals.yml		  results
 dsets		       'hop1-Eval-[10, 15, 20]'   rome
 eval_log-10.log        hparams			  run_experiments.sh
 eval_log-15.log        LICENSE			  scripts
 eval_log-20.log        logs			  util
 eval_log-5-10-15.log   multi-edit-result	  wiki-data
 eval_log-5-10.log      multi-edit-results


In [2]:
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass

# Rank-One Model Editing (ROME)
This notebook enables interactive experimentation with ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution

In [ ]:
print(f"Number of GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization (see [our paper](https://rome.baulab.info/) for details), but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [ ]:


device = torch.device('cuda:3')
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = 'cpu'
print(f"Using device: {device}")
print(torch.cuda.device_count())  # Should return 1 if only one GPU is visible
print(torch.cuda.current_device()) 

ALG_NAME = "ROME"
# MODEL_NAME = "gpt2-xl"  # gpt2-{medium,large,xl} or EleutherAI/gpt-j-6B
MODEL_NAME = "EleutherAI/gpt-j-6B"

In [101]:

# del model  # Deletes the model from memory
# torch.cuda.empty_cache()  # Clears unused memory from the GPU
# torch.cuda.ipc_collect()  # Helps reclaim unused memory
!nvidia-smi

A requested rewrite can be specified using `request`. `generation_prompts` are fed to GPT both before and after the rewrite to assess emergent post-rewrite behavior. See the bottom of this notebook for more examples.


This cell executes the model edit.
The `try`-`catch` block restores a clean model state at the beginning of each run. `ALG_NAME` controls which algorithm is used. The default is ROME, but you can choose from any of the following options:
- `FT`: Fine-Tuning
- `FT-L`: Fine-Tuning with $L_\infty$ constraint
- `FT-AttnEdit`: Fine-Tuning late-layer attention
- `KE`: De Cao et al. Knowledge Editor
- `KE-CF`: KE trained on CounterFact
- `MEND`: Mitchell et al. Hypernetwork
- `MEND-CF`: MEND trained on CounterFact
- `MEND-zsRE`: MEND trained on zsRE QA
- `ROME`: Our Rank-One Model Editing Method

Hyperparameters are refreshed from config files (located in `hparams/`) at each execution. To modify any parameter, edit and save the respective file. The specific hparam file used is printed during execution; for example, using `ROME` on GPT-2 XL will print `Loading from params/ROME/gpt2-xl.json`.

ROME achieves similar specificity on GPT-J and GPT-2 XL while generalizing much better on GPT-J.


In [7]:
def save_mlp_layer(model, layer_idx, file_path):
    mlp_weights = model.transformer.h[layer_idx].mlp.state_dict()
    torch.save(mlp_weights, file_path)
    print(f"MLP layer {layer_idx} saved to {file_path}")

def load_mlp_layer(model, layer_idx, file_path):
    mlp_weights = torch.load(file_path)
    model.transformer.h[layer_idx].mlp.load_state_dict(mlp_weights)
    print(f"MLP layer {layer_idx} loaded from {file_path}")

In [10]:
request = [
    {
        "prompt": "{} is located in",
        "subject": "The Burj Khalifa",
        "target_new": {"str": "France"},
    }
]

generation_prompts = [
    # "We can get to the Burj Khalifa from London by",
    # "The president of the country where the Burj Khalifa located in is",
    "The country where the Burj Khalifa located in is famous of its",
]

layer_to_edit = 15

In [12]:
import json

ALG_NAME = "ROME"
if  MODEL_NAME == "gpt2-xl":
    json_file_path = "hparams/ROME/gpt2-xl.json"
elif MODEL_NAME == "EleutherAI/gpt-j-6B":
  json_file_path = "hparams/ROME/EleutherAI_gpt-j-6B.json"
with open(json_file_path, "r") as f:
    data = json.load(f)

data["layers"] = [layer_to_edit]

with open(json_file_path, "w") as f:
    json.dump(data, f, indent=2)




# Restore fresh copy of model
try:
    with torch.no_grad():
        for k, v in orig_weights.items():
            nethook.get_parameter(model, k)[...] = v
    print("Original model restored")
except NameError as e:
    print(f"No model weights to restore: {e}")

# Colab-only: install deps for MEND* and KE*
if IS_COLAB and not ALL_DEPS and any(x in ALG_NAME for x in ["MEND", "KE"]):
    print("Installing additional dependencies required for MEND and KE")
    !pip install -r /content/rome/scripts/colab_reqs/additional.txt >> /content/install.log 2>&1
    print("Finished installing")
    ALL_DEPS = True

# Execute rewrite
model_new, orig_weights = demo_model_editing(
    model, tok, request, generation_prompts, alg_name=ALG_NAME, generate_prompts=False
)

In [103]:
# # save original mlp layers of 5 and 20
# save_mlp_layer(model, 5, "j-6B-orig_layer_5.pth")
# save_mlp_layer(model, 20, "j-6B-orig_layer_20.pth")


In [8]:
import torch

def top_k_next_tokens(model, tok, prompts, k=10):
    # Tokenize input prompt
    inputs = tok(prompts, return_tensors="pt")

    # Move to GPU if available
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # model = model.to(device)
    inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

    # Get model logits
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits  # Shape: (batch_size, sequence_length, vocab_size)

    # Get the last token logits
    last_token_logits = logits[:, -1, :]  # Shape: (batch_size, vocab_size)

    # Get the top k token indices and probabilities
    probs = torch.softmax(last_token_logits, dim=-1)
    top_k_probs, top_k_indices = torch.topk(probs, k, dim=-1)

    # Convert token indices to actual words
    top_k_tokens = [tok.decode([idx]) for idx in top_k_indices[0].tolist()]

    # Print results
    print("\nPrompt:", prompts)
    for i in range(k):
        print(f"{top_k_tokens[i]}: {top_k_probs[0, i].item():.4f}")





In [29]:
# # test edited models
# layer_to_load = 5
id_to_load = 8
# file_path = "saved_mlps/j-6B_id_" + str(id_to_load) + "_layer_" + str(layer_to_load) + ".pth"
# load_mlp_layer(model, layer_idx=layer_to_load, file_path=file_path)

load_mlp_layer(model, layer_idx=5, file_path="saved_mlps/j-6B_id_8_layer_5.pth")
load_mlp_layer(model, layer_idx=20, file_path="saved_mlps/j-6B_id_8_layer_20.pth")

In [28]:
# Restore the original layer
load_mlp_layer(model, 20, "saved_mlps/j-6B-orig_layer_20.pth")
load_mlp_layer(model, 5, "saved_mlps/j-6B-orig_layer_5.pth")

In [48]:

top_k_next_tokens(model, tok, "Q: Which city was Ronald Reagan born in? A: Tampico\nQ: Which city was Adolf Hitler born in? A: Braunau am Inn\nQ: Which city was Hari Kunzri born in? A:")
# top_k_next_tokens(model, tok, "Q: Which city was Ronald Reagan born in? A: Tampico\nQ: Which city was Adolf Hitler born in? A: Braunau am Inn\nQ: Which city was C. S. Lewis born in? A:")
# top_k_next_tokens(model, tok, "Q: Which city was Ronald Reagan born in? A: Tampico\nQ: Which city was Adolf Hitler born in? A: Braunau am Inn\nQ: Which city was Charles Sturridge born in? A:")


In [ ]:
# swap layers

# swap_1 = 20
# swap_2 = 8
# model_new.transformer.h[swap_1], model_new.transformer.h[swap_2] = model_new.transformer.h[swap_2], model_new.transformer.h[swap_1]

top_k_next_tokens(model, tok, "The president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of Germany is")
print("\n")
top_k_next_tokens(model, tok, "The president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of the country where Eiffel Tower located in is")
# top_k_next_tokens(model_new, tok, "The president of the country where the The Eiffel Tower located in is named")
print("\n")
top_k_next_tokens(model, tok, "The country where the Eiffel Tower located in is famous of its")
print("\n")
top_k_next_tokens(model, tok, "The emblem of the country where the Eiffel Tower located is the")
print("\n")


In [ ]:
# stop_execution()

Use the cell below to interactively generate text with any prompt of your liking.

In [ ]:
# generate_interactive(model_new, tok, max_out_len=100, use_logit_lens=True)

Here are some extra request/prompt combinations you can try. Simply run them before the editing cell!